# **Language Model**
ngitung probability kemunculan suatu kalimat, tanpa tau arti dari kalimatnya  
cocok untuk spell correction, speech recognition, sentence formation, word prediction  
  
Kekurangan: butuh data yang sangat besar dan boros computational  
  
**Types of N-gram**
- Unigram: probability berdiri sendiri  
- Bigram: bergantung dengan 1 kata  
- Trigram: probability bergantung sama 2 kata  

Contoh:
Corpus: "Saya makan nasi", "Saya makan roti"

| Bigram | Count | 
|-|-|
| Saya makan | 2 |
| makan nasi | 1 |
| makan roti | 1 |

$P(makan|Saya) = \frac{2}{2} = 1$  
$P(nasi|makan) = \frac{1}{2} = 0.5$

**Data Sparsity** -> kekurangan utama dari Bigram, dia ngga bisa hitung probability untuk kata yang ngga ada di dataset

# Vectorizing

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer  # ubah text ke bentuk vector
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
class NGramLanguageModel:  # pake class biar lebih gampang makenya :)
    def __init__(self, n):  # constructor
        self.n = n
        self.vectorizer = CountVectorizer(
            analyzer='word',  # dia bakal bagi sesuai dengan word, not character
            ngram_range=(n,n)  # fixed size of n
        )
        
    def fit_transform(self, corpus):  # corpus ini yang bakal jadi training data buat di model
        # fit -> load and learn dari data
        # transform -> convert text ke vector
        return self.vectorizer.fit_transform(corpus)
    
    def transform(self, corpus):  # tidak belajar dari vocab baru, ini kalimat yang mau dicari similaritynya dgn dataset
        return self.vectorizer.transform(corpus)

In [3]:
document = [
    'She sells sea shells by the sea shore', 'The shells she sells are surely sea shells', 'So if she sells sea shells on the sea shore'
]

query = 'Surely she sells sea shells'

n = 1

In [4]:
ngram_model = NGramLanguageModel(n)

target_vec = ngram_model.fit_transform(document)
query_vect = ngram_model.transform([query])  # harus pake list biar nggak error

In [5]:
pd.DataFrame(  # yha, basically ini kayak Bag-of-Words
    target_vec.A,  # untuk ngubah matrix jadi array
    columns=ngram_model.vectorizer.get_feature_names_out()  # ngambil fitur jadi columns vocab
)

,are,by,if,on,sea,sells,she,shells,shore,so,surely,the
0,0,1,0,0,2,1,1,1,1,0,0,1
1,1,0,0,0,1,1,1,2,0,0,1,1
2,0,0,1,1,2,1,1,1,1,1,0,1


# Cosine Similarity

In [6]:
similarity = cosine_similarity(query_vect, target_vec).flatten()  # input 2 matrix, diubah jadi 1 Dimensi

data = {'document': document, 'similarity': similarity}
print("Query:", query)
pd.DataFrame(data)

Query: Surely she sells sea shells


,document,similarity
0,She sells sea shells by the sea shore,0.707107
1,The shells she sells are surely sea shells,0.848528
2,So if she sells sea shells on the sea shore,0.645497


# Predict Next Word

In [7]:
from nltk.tokenize import word_tokenize
from collections import Counter

In [8]:
def preprocess(text):
    return word_tokenize(text.lower())

In [9]:
counts = {}  # save next word counts

for sentence in document:
    words = preprocess(sentence)
    for i in range(len(words) - n + 1):
        # saya suka makan nasi goreng (ex: n = 3)
        # saya suka, suka makan, makan nasi, nasi goreng
        # 5 - 2 + 1 = 4
        context = tuple(words[i : i + n - 1])
        # i = 0
        # words[0 : 0 + 3 - 1] = words[0:2] = "saya", "suka"
        next_word = words[i + n - 1]
        # words[0 + 2 - 1] = words[1] = "suka"
        counts.setdefault(context, Counter())[next_word] += 1
        # ("saya",) : Counter({"suka": 1})

In [10]:
query_tokens = preprocess(query)
context = tuple(query_tokens[-(n-1):])
predictions = counts.get(context, Counter()).most_common(5)

In [ ]:
print("Query:", query)
print("Context:", context)
print("Next-word prediction:", predictions)

Query: Surely she sells sea shells
Context: ('surely', 'she', 'sells', 'sea', 'shells')
Next word prediction: []
